<a href="https://colab.research.google.com/github/kratzhar/State-Drug-Utilization-Project/blob/main/Phase4_CIS_660_Group_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# install pyspark in colab
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, IntegerType, LongType,
    DoubleType, StringType, ShortType
)
from pyspark.sql.functions import (
    col, trim, upper, when, lit,
    count, avg, sum as spark_sum, round as spark_round,
    desc, countDistinct,
    monotonically_increasing_id, rank, row_number
)
from pyspark.sql.window import Window

print('Setup done')

Setup done


In [2]:
spark = SparkSession.builder \
    .appName('CIS660_Phase4_Medicaid') \
    .config('spark.sql.legacy.timeParserPolicy', 'LEGACY') \
    .getOrCreate()

print(f'Spark Version: {spark.version}')
print(f'App: {spark.sparkContext.appName}')

Spark Version: 4.0.2
App: CIS660_Phase4_Medicaid


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [34]:
#load csv file to dataframe

sdud_2025 = spark.read.csv('/content/drive/MyDrive/Group Project/StateDrugUtilizationData-2025.csv', header=True)

In [6]:
#save as bronze csv file

output_path = '/content/drive/MyDrive/Group Project/bronze/'
sdud_2025.coalesce(1).write.mode('overwrite').option('header', 'true').csv(output_path)
print(f"DataFrame saved to {output_path}")

DataFrame saved to /content/drive/MyDrive/Group Project/bronze/


## Schema Definition
Defining a StructType schema for the raw CSV. Everything is StringType first
because the raw data has padded zeros in numeric fields.
This matches the staging_raw table from Phase 2.

In [7]:
# all columns as StringType - same as staging_raw in the sql schema
# we'll cast to proper types when building the fact table
staging_schema = StructType([
    StructField('utilization_type',               StringType(), True),
    StructField('state',                          StringType(), True),
    StructField('ndc',                            StringType(), True),
    StructField('labeler_code',                   StringType(), True),
    StructField('product_code',                   StringType(), True),
    StructField('package_size',                   StringType(), True),
    StructField('year',                           StringType(), True),
    StructField('quarter',                        StringType(), True),
    StructField('suppression_used',               StringType(), True),
    StructField('product_name',                   StringType(), True),
    StructField('units_reimbursed',               StringType(), True),
    StructField('number_of_prescriptions',        StringType(), True),
    StructField('total_amount_reimbursed',        StringType(), True),
    StructField('medicaid_amount_reimbursed',     StringType(), True),
    StructField('non_medicaid_amount_reimbursed', StringType(), True)
])

staging_raw = spark.read.csv('/content/drive/MyDrive/Group Project/sdud_2025.csv', header=True, schema=staging_schema)
print(f'Staging raw loaded: {staging_raw.count():,} rows')

Staging raw loaded: 3,972,559 rows


In [8]:
# check the schema looks right
print('STAGING_RAW:')
staging_raw.printSchema()

STAGING_RAW:
root
 |-- utilization_type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- ndc: string (nullable = true)
 |-- labeler_code: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- package_size: string (nullable = true)
 |-- year: string (nullable = true)
 |-- quarter: string (nullable = true)
 |-- suppression_used: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- units_reimbursed: string (nullable = true)
 |-- number_of_prescriptions: string (nullable = true)
 |-- total_amount_reimbursed: string (nullable = true)
 |-- medicaid_amount_reimbursed: string (nullable = true)
 |-- non_medicaid_amount_reimbursed: string (nullable = true)



In [9]:
staging_raw.show(5, truncate=False)

+----------------+-----+-----------+------------+------------+------------+----+-------+----------------+------------+----------------+-----------------------+-----------------------+--------------------------+------------------------------+
|utilization_type|state|ndc        |labeler_code|product_code|package_size|year|quarter|suppression_used|product_name|units_reimbursed|number_of_prescriptions|total_amount_reimbursed|medicaid_amount_reimbursed|non_medicaid_amount_reimbursed|
+----------------+-----+-----------+------------+------------+------------+----+-------+----------------+------------+----------------+-----------------------+-----------------------+--------------------------+------------------------------+
|MCOU            |AK   |00002143380|00002       |1433        |80          |2025|3      |true            |TRULICITY   |NULL            |NULL                   |NULL                   |NULL                      |NULL                          |
|FFSU            |AK   |00002143

In [10]:
# quick look at whats in the data
print(f'Distinct states:   {staging_raw.select("state").distinct().count()}')
print(f'Distinct NDCs:     {staging_raw.select("ndc").distinct().count()}')
print(f'Distinct years:    {staging_raw.select("year").distinct().count()}')
print(f'Distinct quarters: {staging_raw.select("quarter").distinct().count()}')
print()
staging_raw.select('year', 'quarter').distinct().orderBy('year', 'quarter').show()

Distinct states:   53
Distinct NDCs:     49033
Distinct years:    1
Distinct quarters: 3

+----+-------+
|year|quarter|
+----+-------+
|2025|      1|
|2025|      2|
|2025|      3|
+----+-------+



Data Cleaning & Dimensional Modeling
Building the 3 normalized tables that match our Phase 2 SQL design:

states — dimension table with distinct state codes
drugs — dimension table with distinct NDCs and drug info
utilization — fact table with prescriptions and reimbursement amounts
Numbers in the CSV are padded with zeros (e.g. "00000

In [11]:
# states dimension - distinct state codes with a surrogate key
# use row_number() to get small sequential IDs (1, 2, 3...) that fit in IntegerType
# monotonically_increasing_id() produces huge values that overflow IntegerType
w_states = Window.orderBy('state_code')
states_df = staging_raw \
    .select(upper(trim(col('state'))).alias('state_code')) \
    .filter(col('state_code').isNotNull() & (col('state_code') != '')) \
    .distinct() \
    .withColumn('state_id', row_number().over(w_states).cast(IntegerType()))

states_df = states_df.cache()
print(f'States: {states_df.count()} rows')
states_df.orderBy('state_code').show(60, truncate=False)

States: 53 rows
+----------+--------+
|state_code|state_id|
+----------+--------+
|AK        |1       |
|AL        |2       |
|AR        |3       |
|AZ        |4       |
|CA        |5       |
|CO        |6       |
|CT        |7       |
|DC        |8       |
|DE        |9       |
|FL        |10      |
|GA        |11      |
|HI        |12      |
|IA        |13      |
|ID        |14      |
|IL        |15      |
|IN        |16      |
|KS        |17      |
|KY        |18      |
|LA        |19      |
|MA        |20      |
|MD        |21      |
|ME        |22      |
|MI        |23      |
|MN        |24      |
|MO        |25      |
|MS        |26      |
|MT        |27      |
|NC        |28      |
|ND        |29      |
|NE        |30      |
|NH        |31      |
|NJ        |32      |
|NM        |33      |
|NV        |34      |
|NY        |35      |
|OH        |36      |
|OK        |37      |
|OR        |38      |
|PA        |39      |
|PR        |40      |
|RI        |41      |
|SC        |42  

In [12]:
# drugs dimension - one row per unique NDC
# package_size: only cast to int if its a clean number
drugs_df = staging_raw \
    .filter(col('ndc').isNotNull() & (trim(col('ndc')) != '')) \
    .select(
        trim(col('ndc')).alias('ndc'),
        trim(col('labeler_code')).alias('labeler_code'),
        trim(col('product_code')).alias('product_code'),
        when(col('package_size').rlike('^[0-9]+$'),
             col('package_size').cast(IntegerType()))
            .otherwise(lit(None).cast(IntegerType())).alias('package_size'),
        trim(col('product_name')).alias('product_name')
    ) \
    .dropDuplicates(['ndc']) \
    .withColumn('drug_id', monotonically_increasing_id())

drugs_df = drugs_df.cache()
print(f'Drugs: {drugs_df.count():,} rows')
drugs_df.show(10, truncate=False)

Drugs: 49,033 rows
+-----------+------------+------------+------------+------------+-------+
|ndc        |labeler_code|product_code|package_size|product_name|drug_id|
+-----------+------------+------------+------------+------------+-------+
|00008049801|00008       |0498        |1           |PHENERGAN   |0      |
|00069030601|00069       |0306        |1           |TRAZIMERA   |1      |
|00069246519|00069       |2465        |19          |ABRYSVO     |2      |
|00071080340|00071       |0803        |40          |NEURONTIN   |3      |
|00074116301|00074       |1163        |1           |BUPV 0.5%   |4      |
|00074611413|00074       |6114        |13          |DEPAKOTE S  |5      |
|00078038334|00078       |0383        |34          |DIOVAN HCT  |6      |
|00078043905|00078       |0439        |5           |RITALIN 5   |7      |
|00078097989|00078       |0979        |89          |MAYZENT FC  |8      |
|00093555201|00093       |5552        |1           |DEXMETHYLP  |9      |
+-----------+------

In [13]:
# utilization fact table
# join staging with states and drugs to get surrogate keys
# cast the padded zero strings to actual numbers
utilization_df = staging_raw \
    .filter(
        col('state').isNotNull() & (trim(col('state')) != '') &
        col('ndc').isNotNull() & (trim(col('ndc')) != '') &
        col('year').isNotNull() & col('quarter').isNotNull()
    ) \
    .join(states_df, upper(trim(staging_raw['state'])) == states_df['state_code']) \
    .join(drugs_df, trim(staging_raw['ndc']) == drugs_df['ndc']) \
    .select(
        states_df['state_id'],
        drugs_df['drug_id'],
        staging_raw['year'].cast(ShortType()).alias('year'),
        staging_raw['quarter'].cast(ShortType()).alias('quarter'),
        staging_raw['number_of_prescriptions'].cast(LongType()).alias('number_of_prescriptions'),
        staging_raw['total_amount_reimbursed'].cast(DoubleType()).alias('total_amount_reimbursed'),
        staging_raw['units_reimbursed'].cast(DoubleType()).cast(LongType()).alias('units_reimbursed')
    )

In [14]:
# print schemas for all 3 tables to verify types
print('STATES:'); states_df.printSchema()
print('DRUGS:'); drugs_df.printSchema()
print('UTILIZATION:'); utilization_df.printSchema()

STATES:
root
 |-- state_code: string (nullable = true)
 |-- state_id: integer (nullable = false)

DRUGS:
root
 |-- ndc: string (nullable = true)
 |-- labeler_code: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- package_size: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- drug_id: long (nullable = false)

UTILIZATION:
root
 |-- state_id: integer (nullable = false)
 |-- drug_id: long (nullable = false)
 |-- year: short (nullable = true)
 |-- quarter: short (nullable = true)
 |-- number_of_prescriptions: long (nullable = true)
 |-- total_amount_reimbursed: double (nullable = true)
 |-- units_reimbursed: long (nullable = true)



In [15]:
states_df.show(5, truncate=False)
drugs_df.show(5, truncate=False)
utilization_df.show(5, truncate=False)

+----------+--------+
|state_code|state_id|
+----------+--------+
|AK        |1       |
|AL        |2       |
|AR        |3       |
|AZ        |4       |
|CA        |5       |
+----------+--------+
only showing top 5 rows
+-----------+------------+------------+------------+------------+-------+
|ndc        |labeler_code|product_code|package_size|product_name|drug_id|
+-----------+------------+------------+------------+------------+-------+
|00008049801|00008       |0498        |1           |PHENERGAN   |0      |
|00069030601|00069       |0306        |1           |TRAZIMERA   |1      |
|00069246519|00069       |2465        |19          |ABRYSVO     |2      |
|00071080340|00071       |0803        |40          |NEURONTIN   |3      |
|00074116301|00074       |1163        |1           |BUPV 0.5%   |4      |
+-----------+------------+------------+------------+------------+-------+
only showing top 5 rows
+--------+------------+----+-------+-----------------------+-----------------------+----

In [16]:
# data quality checks
print('Row counts:')
print(f'  staging_raw:  {staging_raw.count():>12,}')
print(f'  states:       {states_df.count():>12,}')
print(f'  drugs:        {drugs_df.count():>12,}')
print(f'  utilization:  {utilization_df.count():>12,}')
print()
neg_reimb = utilization_df.filter(col('total_amount_reimbursed') < 0).count()
neg_rx = utilization_df.filter(col('number_of_prescriptions') < 0).count()
null_year = utilization_df.filter(col('year').isNull()).count()
null_qtr = utilization_df.filter(col('quarter').isNull()).count()
print(f'Negative reimbursements: {neg_reimb}')
print(f'Negative prescriptions:  {neg_rx}')
print(f'Null years:              {null_year}')
print(f'Null quarters:           {null_qtr}')

Row counts:
  staging_raw:     3,972,559
  states:                 53
  drugs:              49,033
  utilization:     3,972,559

Negative reimbursements: 0
Negative prescriptions:  0
Null years:              0
Null quarters:           0


In [17]:
# duplicate check on the grain (same state + drug + year + quarter showing up more than once)
dupes = utilization_df.groupBy('state_id', 'drug_id', 'year', 'quarter') \
    .agg(count('*').alias('cnt')) \
    .filter(col('cnt') > 1)
print(f'Duplicate grain combos: {dupes.count()}')
if dupes.count() > 0:
    dupes.orderBy(desc('cnt')).show(10)
else:
    print('No duplicates - grain is clean')

Duplicate grain combos: 1015400
+--------+-------------+----+-------+---+
|state_id|      drug_id|year|quarter|cnt|
+--------+-------------+----+-------+---+
|       1|1108101562377|2025|      3|  2|
|       5|  85899346171|2025|      3|  2|
|       1| 601295421528|2025|      2|  2|
|       5|  85899346179|2025|      3|  2|
|      11| 128849019110|2025|      1|  2|
|       5|  85899346135|2025|      2|  2|
|      15| 128849019101|2025|      3|  2|
|       9|1194000908533|2025|      3|  2|
|      18| 317827580141|2025|      2|  2|
|      18| 850403524876|2025|      3|  2|
+--------+-------------+----+-------+---+
only showing top 10 rows


Saving cleaned tables as csv (silver)

In [18]:
# silver layer - cleaned tables as csv
states_df.coalesce(1).write.mode('overwrite').option('header', 'true').csv('/content/drive/MyDrive/Group Project/silver/states')
drugs_df.coalesce(1).write.mode('overwrite').option('header', 'true').csv('/content/drive/MyDrive/Group Project/silver/drugs')
utilization_df.coalesce(1).write.mode('overwrite').option('header', 'true').csv('/content/drive/MyDrive/Group Project/silver/utilization')
print('Silver layer saved:')


Silver layer saved:


In [19]:
!ls '/content/drive/MyDrive/Group Project/silver/'

drugs  states  utilization


## Transformations & Spark SQL
Running the 10 analytics queries from Phase 2 in Spark.
Each query uses DataFrame API or Spark SQL (some show both approaches).

In [20]:
# register everything as temp views for SQL queries
states_df.createOrReplaceTempView('states')
drugs_df.createOrReplaceTempView('drugs')
utilization_df.createOrReplaceTempView('utilization')

spark.sql('SHOW TABLES').show()

+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|         |      drugs|       true|
|         |     states|       true|
|         |utilization|       true|
+---------+-----------+-----------+



In [21]:
# Q1: top 20 states by total medicaid reimbursement (multi-table join)
print('Q1 - DataFrame API: Top 20 States by Total Reimbursement')
print()
utilization_df.join(states_df, 'state_id') \
    .groupBy('state_code') \
    .agg(
        spark_round(spark_sum('total_amount_reimbursed'), 2).alias('total_reimbursement'),
        spark_sum('number_of_prescriptions').alias('total_prescriptions'),
        spark_sum('units_reimbursed').alias('total_units')
    ) \
    .orderBy(desc('total_reimbursement')) \
    .limit(20).show(20, truncate=False)

Q1 - DataFrame API: Top 20 States by Total Reimbursement

+----------+-------------------+-------------------+-----------+
|state_code|total_reimbursement|total_prescriptions|total_units|
+----------+-------------------+-------------------+-----------+
|XX        |8.737000743475E10  |544593441          |35201877948|
|CA        |1.353157445208E10  |67100724           |5076290774 |
|NY        |9.39230636253E9    |53525409           |3534574010 |
|PA        |4.01769125647E9    |25220372           |1568299609 |
|OH        |3.52126778674E9    |27595701           |1775645408 |
|NC        |3.50912882202E9    |16372751           |1205373890 |
|MI        |3.11683586108E9    |21093115           |1148372685 |
|TX        |2.7506166577E9     |19479965           |1496286758 |
|IL        |2.48611217221E9    |17010200           |1061389132 |
|FL        |2.46475127205E9    |15878727           |1033034194 |
|IN        |2.22228138167E9    |13764081           |875225007  |
|KY        |2.11243652735E9    |

In [22]:
# Q1 again in Spark SQL
print('Q1 - Spark SQL: Top 20 States by Total Reimbursement')
print()
spark.sql("""
    SELECT s.state_code,
           ROUND(SUM(u.total_amount_reimbursed), 2) AS total_reimbursement,
           SUM(u.number_of_prescriptions) AS total_prescriptions,
           SUM(u.units_reimbursed) AS total_units
    FROM utilization u
    JOIN states s ON u.state_id = s.state_id
    GROUP BY s.state_code
    ORDER BY total_reimbursement DESC
    LIMIT 20
""").show(20, truncate=False)

Q1 - Spark SQL: Top 20 States by Total Reimbursement

+----------+-------------------+-------------------+-----------+
|state_code|total_reimbursement|total_prescriptions|total_units|
+----------+-------------------+-------------------+-----------+
|XX        |8.737000743475E10  |544593441          |35201877948|
|CA        |1.353157445208E10  |67100724           |5076290774 |
|NY        |9.39230636253E9    |53525409           |3534574010 |
|PA        |4.01769125647E9    |25220372           |1568299609 |
|OH        |3.52126778674E9    |27595701           |1775645408 |
|NC        |3.50912882202E9    |16372751           |1205373890 |
|MI        |3.11683586108E9    |21093115           |1148372685 |
|TX        |2.7506166577E9     |19479965           |1496286758 |
|IL        |2.48611217221E9    |17010200           |1061389132 |
|FL        |2.46475127205E9    |15878727           |1033034194 |
|IN        |2.22228138167E9    |13764081           |875225007  |
|KY        |2.11243652735E9    |2032

In [23]:
# Q2: annual spend and prescription volume by state (GROUP BY + aggregation)
print('Q2 - DataFrame API: Annual Spend by State')
utilization_df.join(states_df, 'state_id') \
    .groupBy('state_code', 'year') \
    .agg(
        spark_round(spark_sum('total_amount_reimbursed'), 2).alias('annual_spend'),
        spark_sum('number_of_prescriptions').alias('annual_prescriptions'),
        spark_round(
            spark_sum(col('total_amount_reimbursed')) / spark_sum(col('number_of_prescriptions')), 2
        ).alias('avg_cost_per_rx')
    ).orderBy('state_code', 'year').show(30)

# Q3: states where total reimbursement exceeded $1 billion (HAVING clause)
print('Q3 - Spark SQL: States Over $1 Billion')
spark.sql("""
    SELECT s.state_code,
           ROUND(SUM(u.total_amount_reimbursed), 2) AS total_spend
    FROM utilization u
    JOIN states s ON u.state_id = s.state_id
    GROUP BY s.state_code
    HAVING SUM(u.total_amount_reimbursed) > 1000000000
    ORDER BY total_spend DESC
""").show(truncate=False)

Q2 - DataFrame API: Annual Spend by State
+----------+----+-----------------+--------------------+---------------+
|state_code|year|     annual_spend|annual_prescriptions|avg_cost_per_rx|
+----------+----+-----------------+--------------------+---------------+
|        AK|2025|    1.433889518E8|              804570|         178.22|
|        AL|2025|   7.7767514004E8|             4780044|         162.69|
|        AR|2025|   3.7232954192E8|             4051155|          91.91|
|        AZ|2025|  1.55125513928E9|            10829691|         143.24|
|        CA|2025|1.353157445208E10|            67100724|         201.66|
|        CO|2025|  1.23140613554E9|             7243882|         169.99|
|        CT|2025|  1.49961743355E9|             6833026|         219.47|
|        DC|2025|   2.5071501958E8|             1400095|         179.07|
|        DE|2025|   2.0487231487E8|             1722019|         118.97|
|        FL|2025|  2.46475127205E9|            15878727|         155.22|
|        

In [24]:
# Q4: drugs whose total reimbursement exceeds the platform-wide average (subquery logic)
print('Q4 - DataFrame API: Drugs Above Average Reimbursement')
drug_totals = utilization_df.groupBy('drug_id').agg(
    spark_sum('total_amount_reimbursed').alias('total_reimbursement')
)
platform_avg = drug_totals.agg(avg('total_reimbursement')).collect()[0][0]
print(f'Platform avg per drug: ${platform_avg:,.2f}')
drug_totals.join(drugs_df, 'drug_id') \
    .filter(col('total_reimbursement') > platform_avg) \
    .select('ndc', spark_round('total_reimbursement', 2).alias('total_reimbursement')) \
    .orderBy(desc('total_reimbursement')).limit(25).show(25, truncate=False)

# Q5: classify drugs into cost tiers (CASE logic)
print('Q5 - Spark SQL: Drug Cost Tiers')
spark.sql("""
    SELECT d.ndc, d.product_name,
           SUM(u.number_of_prescriptions) AS total_rx,
           ROUND(SUM(u.total_amount_reimbursed), 2) AS total_reimbursement,
           CASE
               WHEN SUM(u.total_amount_reimbursed) /
                    NULLIF(SUM(u.number_of_prescriptions), 0) > 1000 THEN 'Ultra High Cost'
               WHEN SUM(u.total_amount_reimbursed) /
                    NULLIF(SUM(u.number_of_prescriptions), 0) > 100  THEN 'Expensive'
               WHEN SUM(u.total_amount_reimbursed) /
                    NULLIF(SUM(u.number_of_prescriptions), 0) > 10   THEN 'Moderate'
               ELSE 'Affordable'
           END AS cost_category
    FROM utilization u
    JOIN drugs d ON u.drug_id = d.drug_id
    GROUP BY d.ndc, d.product_name
    HAVING SUM(u.number_of_prescriptions) > 1000
    ORDER BY total_reimbursement DESC
    LIMIT 25
""").show(25, truncate=False)

Q4 - DataFrame API: Drugs Above Average Reimbursement
Platform avg per drug: $4,716,057.39
+-----------+-------------------+
|ndc        |total_reimbursement|
+-----------+-------------------+
|61958250101|5.06984441207E9    |
|00074055402|3.49365144926E9    |
|00024591502|2.72183830824E9    |
|00169418113|2.07575548656E9    |
|50458056401|1.91786657542E9    |
|57894006103|1.90465091063E9    |
|00169477212|1.86371636474E9    |
|00003089421|1.8413991638E9     |
|00597015330|1.7465562552E9     |
|00169413013|1.72034123363E9    |
|00074210001|1.57068667714E9    |
|00597015230|1.54995107684E9    |
|51167033101|1.54210251433E9    |
|58406003204|1.36066348941E9    |
|00169452414|1.05689000246E9    |
|61874011530|1.04470106066E9    |
|00310621030|1.0444389632E9     |
|00078107068|1.04326893414E9    |
|12496120803|9.9002141896E8     |
|72618300002|9.3232597033E8     |
|00074262528|8.7640615084E8     |
|00002143480|8.7042189767E8     |
|00074012402|8.5905984111E8     |
|00024591401|8.4978340417

In [25]:
# Q4: drugs whose total reimbursement exceeds the platform-wide average (subquery logic)
print('Q4 - DataFrame API: Drugs Above Average Reimbursement')
drug_totals = utilization_df.groupBy('drug_id').agg(
    spark_sum('total_amount_reimbursed').alias('total_reimbursement')
)
platform_avg = drug_totals.agg(avg('total_reimbursement')).collect()[0][0]
print(f'Platform avg per drug: ${platform_avg:,.2f}')
drug_totals.join(drugs_df, 'drug_id') \
    .filter(col('total_reimbursement') > platform_avg) \
    .select('ndc', spark_round('total_reimbursement', 2).alias('total_reimbursement')) \
    .orderBy(desc('total_reimbursement')).limit(25).show(25, truncate=False)

# Q5: classify drugs into cost tiers (CASE logic)
print('Q5 - Spark SQL: Drug Cost Tiers')
spark.sql("""
    SELECT d.ndc, d.product_name,
           SUM(u.number_of_prescriptions) AS total_rx,
           ROUND(SUM(u.total_amount_reimbursed), 2) AS total_reimbursement,
           CASE
               WHEN SUM(u.total_amount_reimbursed) /
                    NULLIF(SUM(u.number_of_prescriptions), 0) > 1000 THEN 'Ultra High Cost'
               WHEN SUM(u.total_amount_reimbursed) /
                    NULLIF(SUM(u.number_of_prescriptions), 0) > 100  THEN 'Expensive'
               WHEN SUM(u.total_amount_reimbursed) /
                    NULLIF(SUM(u.number_of_prescriptions), 0) > 10   THEN 'Moderate'
               ELSE 'Affordable'
           END AS cost_category
    FROM utilization u
    JOIN drugs d ON u.drug_id = d.drug_id
    GROUP BY d.ndc, d.product_name
    HAVING SUM(u.number_of_prescriptions) > 1000
    ORDER BY total_reimbursement DESC
    LIMIT 25
""").show(25, truncate=False)

Q4 - DataFrame API: Drugs Above Average Reimbursement
Platform avg per drug: $4,716,057.39
+-----------+-------------------+
|ndc        |total_reimbursement|
+-----------+-------------------+
|61958250101|5.06984441207E9    |
|00074055402|3.49365144926E9    |
|00024591502|2.72183830824E9    |
|00169418113|2.07575548656E9    |
|50458056401|1.91786657542E9    |
|57894006103|1.90465091063E9    |
|00169477212|1.86371636474E9    |
|00003089421|1.8413991638E9     |
|00597015330|1.7465562552E9     |
|00169413013|1.72034123363E9    |
|00074210001|1.57068667714E9    |
|00597015230|1.54995107684E9    |
|51167033101|1.54210251433E9    |
|58406003204|1.36066348941E9    |
|00169452414|1.05689000246E9    |
|61874011530|1.04470106066E9    |
|00310621030|1.0444389632E9     |
|00078107068|1.04326893414E9    |
|12496120803|9.9002141896E8     |
|72618300002|9.3232597033E8     |
|00074262528|8.7640615084E8     |
|00002143480|8.7042189767E8     |
|00074012402|8.5905984111E8     |
|00024591401|8.4978340417

In [26]:
# Q7: top 10 states by total prescription volume (multi-join + group by)
print('Q7 - DataFrame API: Top 10 States by Prescription Volume')
utilization_df.join(states_df, 'state_id') \
    .groupBy('state_code') \
    .agg(spark_sum('number_of_prescriptions').alias('total_prescriptions')) \
    .orderBy(desc('total_prescriptions')).limit(10).show(10, truncate=False)

# Q8: high volume drugs classified by cost per rx (HAVING + CASE + multi-join)
print('Q8 - Spark SQL: High-Volume Drug Classification')
spark.sql("""
    SELECT d.ndc, d.product_name,
           SUM(u.number_of_prescriptions) AS total_rx,
           ROUND(SUM(u.total_amount_reimbursed), 2) AS total_reimbursement,
           ROUND(SUM(u.total_amount_reimbursed) /
                 NULLIF(SUM(u.number_of_prescriptions), 0), 2) AS avg_cost_per_rx,
           CASE
               WHEN SUM(u.total_amount_reimbursed) /
                    NULLIF(SUM(u.number_of_prescriptions), 0) > 100 THEN 'Expensive'
               ELSE 'Affordable'
           END AS price_category
    FROM utilization u
    JOIN drugs d ON u.drug_id = d.drug_id
    GROUP BY d.ndc, d.product_name
    HAVING SUM(u.number_of_prescriptions) > 1000
    ORDER BY total_reimbursement DESC
    LIMIT 25
""").show(25, truncate=False)

Q7 - DataFrame API: Top 10 States by Prescription Volume
+----------+-------------------+
|state_code|total_prescriptions|
+----------+-------------------+
|XX        |544593441          |
|CA        |67100724           |
|NY        |53525409           |
|OH        |27595701           |
|PA        |25220372           |
|MI        |21093115           |
|KY        |20326271           |
|TX        |19479965           |
|IL        |17010200           |
|NC        |16372751           |
+----------+-------------------+

Q8 - Spark SQL: High-Volume Drug Classification
+-----------+------------+--------+-------------------+---------------+--------------+
|ndc        |product_name|total_rx|total_reimbursement|avg_cost_per_rx|price_category|
+-----------+------------+--------+-------------------+---------------+--------------+
|61958250101|BIKTARVY 5  |1115967 |5.06984441207E9    |4543.01        |Expensive     |
|00074055402|HUMIRA(CF)  |425906  |3.49365144926E9    |8202.87        |Expensive    

In [27]:
# Q9: quarterly trend analysis
print('Q9 - DataFrame API: Quarterly Trends')
quarterly_trends = utilization_df.groupBy('quarter') \
    .agg(
        spark_sum('number_of_prescriptions').alias('total_prescriptions'),
        spark_round(spark_sum('total_amount_reimbursed'), 2).alias('total_spend'),
        spark_round(
            spark_sum(col('total_amount_reimbursed')) / spark_sum(col('number_of_prescriptions')), 2
        ).alias('avg_cost_per_rx')
    ).orderBy('quarter')
quarterly_trends.show()

print('Q9 - Spark SQL: Quarterly Trends')
spark.sql("""
    SELECT quarter,
           SUM(number_of_prescriptions) AS total_prescriptions,
           ROUND(SUM(total_amount_reimbursed), 2) AS total_spend,
           ROUND(SUM(total_amount_reimbursed) /
                 NULLIF(SUM(number_of_prescriptions), 0), 2) AS avg_cost_per_rx
    FROM utilization
    GROUP BY quarter
    ORDER BY quarter
""").show()

Q9 - DataFrame API: Quarterly Trends
+-------+-------------------+-----------------+---------------+
|quarter|total_prescriptions|      total_spend|avg_cost_per_rx|
+-------+-------------------+-----------------+---------------+
|      1|          366414922|5.450417489466E10|         148.75|
|      2|          354682343|5.514249243968E10|         155.47|
|      3|          361379570|5.891937202586E10|         163.04|
+-------+-------------------+-----------------+---------------+

Q9 - Spark SQL: Quarterly Trends
+-------+-------------------+-----------------+---------------+
|quarter|total_prescriptions|      total_spend|avg_cost_per_rx|
+-------+-------------------+-----------------+---------------+
|      1|          366414922|5.450417489466E10|         148.75|
|      2|          354682343|5.514249243968E10|         155.47|
|      3|          361379570|5.891937202586E10|         163.04|
+-------+-------------------+-----------------+---------------+



In [28]:
# Q10: top cost-driver drug in each state (RANK window function + subquery)
print('Q10 - Spark SQL: Top Cost-Driver Drug Per State')
spark.sql("""
    SELECT state_code, ndc, product_name,
           total_reimbursed, state_rank
    FROM (
        SELECT s.state_code, d.ndc, d.product_name,
               ROUND(SUM(u.total_amount_reimbursed), 2) AS total_reimbursed,
               RANK() OVER (
                   PARTITION BY s.state_code
                   ORDER BY SUM(u.total_amount_reimbursed) DESC
               ) AS state_rank
        FROM utilization u
        JOIN drugs d ON u.drug_id = d.drug_id
        JOIN states s ON u.state_id = s.state_id
        GROUP BY s.state_code, d.ndc, d.product_name
    ) ranked
    WHERE state_rank = 1
    ORDER BY total_reimbursed DESC
""").show(50, truncate=False)

Q10 - Spark SQL: Top Cost-Driver Drug Per State
+----------+-----------+------------+----------------+----------+
|state_code|ndc        |product_name|total_reimbursed|state_rank|
+----------+-----------+------------+----------------+----------+
|XX        |61958250101|BIKTARVY 5  |2.53513775583E9 |1         |
|NY        |61958250101|BIKTARVY 5  |5.7345990287E8  |1         |
|CA        |61958250101|BIKTARVY 5  |3.7172340898E8  |1         |
|PA        |61958250101|BIKTARVY 5  |1.3048562307E8  |1         |
|NC        |61958250101|BIKTARVY 5  |1.1488760801E8  |1         |
|IL        |61958250101|BIKTARVY 5  |1.0803087232E8  |1         |
|MD        |61958250101|BIKTARVY 5  |8.846847215E7   |1         |
|OH        |00074055402|HUMIRA(CF)  |8.459795798E7   |1         |
|WI        |00074055402|HUMIRA(CF)  |8.233774532E7   |1         |
|MI        |00074055402|HUMIRA(CF)  |8.223073528E7   |1         |
|LA        |61958250101|BIKTARVY 5  |7.271871191E7   |1         |
|FL        |61958250101|BIKT

Saving Aggregated Summary Tables as Parquet (Gold)

In [29]:
# state summary
gold_state_summary = utilization_df.join(states_df, 'state_id') \
    .groupBy('state_id', 'state_code') \
    .agg(
        spark_round(spark_sum('total_amount_reimbursed'), 2).alias('total_reimbursement'),
        spark_sum('number_of_prescriptions').alias('total_prescriptions'),
        spark_sum('units_reimbursed').alias('total_units'),
        countDistinct('drug_id').alias('unique_drugs')
    )
gold_state_summary.write.mode('overwrite').parquet('/content/drive/MyDrive/Group Project/gold/state_summary')
print(f'state_summary: {gold_state_summary.count():,} rows')

# drug summary
gold_drug_summary = utilization_df.join(drugs_df, 'drug_id') \
    .groupBy('drug_id', 'ndc', 'product_name') \
    .agg(
        spark_round(spark_sum('total_amount_reimbursed'), 2).alias('total_reimbursement'),
        spark_sum('number_of_prescriptions').alias('total_prescriptions'),
        countDistinct('state_id').alias('num_states')
    )
gold_drug_summary.write.mode('overwrite').parquet('/content/drive/MyDrive/Group Project/gold/drug_summary')
print(f'drug_summary: {gold_drug_summary.count():,} rows')

# quarterly trends
gold_quarterly = utilization_df.groupBy('year', 'quarter') \
    .agg(
        spark_sum('number_of_prescriptions').alias('total_prescriptions'),
        spark_round(spark_sum('total_amount_reimbursed'), 2).alias('total_spend'),
        countDistinct('drug_id').alias('unique_drugs'),
        countDistinct('state_id').alias('unique_states')
    ).orderBy('year', 'quarter')
gold_quarterly.write.mode('overwrite').parquet('/content/drive/MyDrive/Group Project/gold/quarterly_trends')
print(f'quarterly_trends: {gold_quarterly.count():,} rows')

state_summary: 53 rows
drug_summary: 49,033 rows
quarterly_trends: 3 rows


In [30]:
# read back from parquet to make sure it saved right
vs = spark.read.parquet('/content/drive/MyDrive/Group Project/gold/state_summary')
print(f'state_summary: {vs.count():,} rows')
vs.printSchema()
vs.show(5, truncate=False)

vd = spark.read.parquet('/content/drive/MyDrive/Group Project/gold/drug_summary')
print(f'drug_summary: {vd.count():,} rows')
vd.show(5, truncate=False)

vq = spark.read.parquet('/content/drive/MyDrive/Group Project/gold/quarterly_trends')
print(f'quarterly_trends: {vq.count():,} rows')
vq.show(5, truncate=False)

state_summary: 53 rows
root
 |-- state_id: integer (nullable = true)
 |-- state_code: string (nullable = true)
 |-- total_reimbursement: double (nullable = true)
 |-- total_prescriptions: long (nullable = true)
 |-- total_units: long (nullable = true)
 |-- unique_drugs: long (nullable = true)

+--------+----------+-------------------+-------------------+-----------+------------+
|state_id|state_code|total_reimbursement|total_prescriptions|total_units|unique_drugs|
+--------+----------+-------------------+-------------------+-----------+------------+
|28      |NC        |3.50912882202E9    |16372751           |1205373890 |26464       |
|20      |MA        |1.90344964251E9    |10761937           |774502427  |26714       |
|46      |UT        |3.896432453E8      |2523791            |277519450  |18295       |
|19      |LA        |1.86602878575E9    |13333463           |638321731  |22842       |
|22      |ME        |3.9111706974E8     |2077835            |156892126  |16675       |
+--------